# Memanggil library yang akan di pakai

In [17]:
import os
import cv2
import numpy as np
import pandas as pd
from skimage.feature import graycomatrix, graycoprops

# Direktori dari data biar tidak keliru bisa kalian sesuaikan

In [4]:
# ini direktori nya
BASE_DIR = "../data/"

# di dalam direktori ada sub folder baru yaitu berantakan dan rapi
CLASSES = ["berantakan", "rapi"]

# Untuk ekstraksi fitur mean RGB serta MEAN std

In [ ]:
data_fitur = []

for label in CLASSES:
  folder_path = os.path.join(BASE_DIR, label)

  file_list = os.listdir(folder_path)

  for filename in file_list:
    if not filename.lower().endswith((".jpg", ".jpeg", ".png", ".bmp", ".webp")):
      continue

    img_path = os.path.join(folder_path, filename)
    img = cv2.imread(img_path)

    if img is None:
      print(f"Gagal membaca gambar : {img_path}")
      continue

    # dijadikan grayscale buat penambahan fitur
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)

    # laplacian mengukur kerumitan detail garis
    laplacian_var = cv2.Laplacian(gray, cv2.CV_64F).var()

    # canny edge mengukur seberapa banyak garis tepi  di gambar
    edges = cv2.Canny(gray, 100, 200)
    edge_density = np.sum(edges > 0) / edges.size

    # menghitung matriks GLCM
    glcm = graycomatrix(gray, distances=[1], angles=[0], levels=256, symmetric=True, normed=True)

    # megambil 4 nilai statistik dari GLCM
    glcm_contrast = graycoprops(glcm, 'contrast')[0, 0]
    glcm_homogeneity = graycoprops(glcm, 'homogeneity')[0, 0]
    glcm_energy = graycoprops(glcm, 'energy')[0, 0]
    glcm_correlation = graycoprops(glcm, 'correlation')[0, 0]

    b_channel = img[:, :, 0]
    g_channel = img[:, :, 1]
    r_channel = img[:, :, 2]

    mean_r = np.mean(r_channel)
    std_r = np.std(r_channel)

    mean_g = np.mean(g_channel)
    std_g = np.std(g_channel)

    mean_b = np.mean(b_channel)
    std_b = np.std(b_channel)

    data_fitur.append({
      "filename" : filename,
      "label" : label,

      # fitur untuk warna
      "mean_red":round(mean_r, 4),
      "mean_green":round(mean_g, 4),
      "mean_blue":round(mean_b, 4),
      "std_red":round(std_r, 4),
      "std_green":round(std_g, 4),
      "std_blue":round(std_b, 4),
      
      # fitur untuk tepi
      "edge_laplacian_variance":round(laplacian_var, 4),
      "edge_density":round(edge_density, 4),

      # fitur untuk tekstur GLCM
      "glcm_contrast":round(glcm_contrast, 4),
      "glcm_homogeneity":round(glcm_homogeneity, 4),
      "glcm_energy":round(glcm_energy, 4),
      "glcm_correlation":round(glcm_correlation, 4),
    })

## Setelah sukses ekstraksi fitur maka tampilkan 5 data awal

In [19]:
df = pd.DataFrame(data_fitur)
print(df.head())

             filename       label  mean_red  mean_greed  mean_blue  std_red  \
0  berantakan_001.jpg  berantakan  133.0250    121.0971   109.7106  55.4688   
1  berantakan_002.jpg  berantakan  117.2852    109.3012    99.4207  64.9177   
2  berantakan_003.jpg  berantakan  160.6719    142.4771   134.9819  47.2719   
3  berantakan_004.jpg  berantakan  151.9835    138.3797   131.6321  62.3337   
4  berantakan_005.jpg  berantakan  133.8936    112.9759   103.2898  53.9008   

   std_green  std_blue  edge_laplacian_variance  edge_density  glcm_contrast  \
0    53.8199   54.1130                 503.4941        0.0515       103.5193   
1    61.6087   57.2564                 425.7798        0.0531        95.7767   
2    59.3187   67.8100                  13.6370        0.0099        11.8089   
3    71.1531   74.5744                 212.7481        0.0417        69.2484   
4    57.0202   63.4242                  53.3302        0.0181        30.1308   

   glcm_homogeneity  glcm_energy  glcm_corre

## Simpan ke file csv

In [20]:
output_csv = "../hasil_ekstraksi/fitur_rgb_std.csv"
df.to_csv(output_csv, index=False)